# 02 - Data Verification Layer: Rule-Based Account & Relevance Filter

Notebook ini memberi label transparan untuk membedakan akun media/aktor politik, institusi pemerintah, komunitas fess pendidikan, fess umum/tidak relevan, dan kandidat warga biasa. Pendekatan rule-based dipakai agar mudah dijustifikasi dalam metodologi esai.

## 1. Load Data

In [ ]:
import ast
import re
from pathlib import Path

import pandas as pd

input_path = Path("raw_tweets_jakarta_banten.csv")
df = pd.read_csv(input_path)

print(f"Loaded: {df.shape[0]:,} rows x {df.shape[1]:,} columns")
df[["author_username", "author_name", "author_followers", "_search_keyword", "_region"]].head()

## 2. Extract Author Metadata

In [ ]:
def parse_author(value):
    if isinstance(value, dict):
        return value
    if pd.isna(value):
        return {}
    try:
        parsed = ast.literal_eval(value)
        return parsed if isinstance(parsed, dict) else {}
    except (ValueError, SyntaxError):
        return {}

authors = df["author"].apply(parse_author) if "author" in df.columns else pd.Series([{}] * len(df))

df["author_username"] = df.get("author_username", pd.Series([None] * len(df))).fillna(authors.apply(lambda x: x.get("userName")))
df["author_name"] = df.get("author_name", pd.Series([None] * len(df))).fillna(authors.apply(lambda x: x.get("name")))
df["author_followers"] = pd.to_numeric(df.get("author_followers", authors.apply(lambda x: x.get("followers"))), errors="coerce")
df["author_following"] = pd.to_numeric(authors.apply(lambda x: x.get("following")), errors="coerce")
df["author_statuses_count"] = pd.to_numeric(authors.apply(lambda x: x.get("statusesCount")), errors="coerce")
df["author_description"] = authors.apply(lambda x: x.get("description", "") or "")
df["author_location"] = authors.apply(lambda x: x.get("location", "") or "")
df["author_is_blue_verified"] = authors.apply(lambda x: bool(x.get("isBlueVerified")))
df["author_is_automated"] = authors.apply(lambda x: bool(x.get("isAutomated")))

df[["author_username", "author_name", "author_followers", "author_following", "author_description"]].head()

## 3. Rule Definitions

In [ ]:
MEDIA_USERNAMES = {
    "sindonews", "detikcom", "kompascom", "tempodotco", "cnnindonesia", "tvonenews",
    "tribunnews", "liputan6dotcom", "kumparan", "antaranews", "metrotv", "beritasatu",
    "republikaonline", "okezonenews", "jawapos", "mediaindonesia", "narasitv",
    "inilahdotcom", "vivacoid", "hariankompas", "pikiran_rakyat", "tempo_plus",
    "mkini_bm", "rmol_id", "alineadotid", "faktacom_", "akuratco", "bisniscom",
    "kompastv", "cnnindonesia", "officialinews_", "beritajakarta",
}

GOV_INSTITUTION_USERNAMES = {
    "dkijakarta", "kemdikdasmen", "kemdikbud_ri", "kemdikbud_ri", "kemendikbud_ri",
    "divhumas_polri", "tmcpoldametro", "univ_indonesia", "ui_official",
    "disdikdki", "disdik_dki", "pemprovbanten", "bantenprov", "dindikbudbanten",
}

POLITICAL_ACTOR_USERNAMES = {
    "pdiperjuangan", "gerindra", "dpp_pkb", "officialpan", "psi_id", "pk_sejahtera",
    "dpppkb", "dppgolkar", "golkar_id", "nasdem", "official_nasdem", "demokrat",
    "pdemokrat", "partaiperindo", "gelora_id", "partaiburuh", "ppppusat",
}

PUBLIC_FIGURE_USERNAMES = {
    "pramonoanung", "agusyudhoyono", "mardanialisera", "fahiraidris", "fadjroel",
    "putuputrayasa", "aniesbaswedan", "ridwankamil", "ganjarpranowo", "sandiuno",
}

EDUCATION_FESS_USERNAMES = {
    "schfess", "schoolfessid", "sbmptnfess", "snbtfess", "utbkfess", "ug_fess",
    "collegemenfess", "campusfess", "univfess", "studytwtfess", "ambisfess",
    "collegemfs", "schcampus",
}

IRRELEVANT_FESS_USERNAMES = {
    "kdrama_menfess", "tubirfess", "worksfess",
}

GENERAL_FESS_USERNAMES = {
    "tanyarlfes", "tanyakanrl", "convomf", "txtdarifess", "askrlfess",
    "bxrwithheld", "nctzenbase", "literarybase",
}

OFFICIAL_DESC_PATTERNS = [
    "akun resmi", "official account", "pemprov", "pemerintah provinsi", "kementerian",
    "dinas pendidikan", "universitas", "university", "polri", "polda",
]

POLITICAL_DESC_PATTERNS = [
    "partai", "fraksi", "dpr", "dprd", "politisi", "anggota dewan", "caleg",
    "legislatif", "parlemen", "relawan", "tim pemenangan",
]

PUBLIC_FIGURE_DESC_PATTERNS = [
    "gubernur", "wakil gubernur", "menteri", "wakil menteri", "anggota dpr",
    "anggota dprd", "senator", "kepala daerah", "juru bicara", "jubir",
]

MEDIA_DESC_PATTERNS = [
    "media", "news", "berita", "portal", "koran", "televisi", "tv", "jurnalisme",
    "newspaper", "online news", "situs berita", "breaking news", "jurnalis",
]

FESS_DESC_PATTERNS = [
    "menfess", "base", "kirim menfess", "auto menfess", "manual menfess",
    "confess", "sender", "submit",
]

EDUCATION_TERMS = [
    "sekolah", "siswa", "murid", "guru", "pendidikan", "ppdb", "spmb", "kjp", "kjmu",
    "bos", "beasiswa", "kuliah", "mahasiswa", "kampus", "ukt", "snbt", "utbk",
    "zonasi", "universitas", "perguruan tinggi", "putus sekolah", "akses sekolah",
]

IRRELEVANT_TERMS = ["kdrama", "drakor", "gosip", "tubir", "idol", "shipper"]

## 4. Classification Functions

In [ ]:
def norm_text(value):
    return "" if pd.isna(value) else str(value).strip().lower()

def contains_any(text, patterns):
    text = norm_text(text)
    return any(pattern in text for pattern in patterns)

def classify_account(row):
    username = norm_text(row.get("author_username"))
    name = norm_text(row.get("author_name"))
    desc = norm_text(row.get("author_description"))
    followers = row.get("author_followers")
    following = row.get("author_following")
    is_automated = bool(row.get("author_is_automated"))
    followers = 0 if pd.isna(followers) else followers
    following = 0 if pd.isna(following) else following

    if username in EDUCATION_FESS_USERNAMES:
        return "education_fess_community"

    if username in IRRELEVANT_FESS_USERNAMES:
        return "irrelevant_fess"

    if username in GENERAL_FESS_USERNAMES or "fess" in username or "menfess" in username or "fess" in name or contains_any(desc, FESS_DESC_PATTERNS):
        return "general_fess"

    if username in GOV_INSTITUTION_USERNAMES:
        return "government_institution"

    if username in PUBLIC_FIGURE_USERNAMES or contains_any(desc, PUBLIC_FIGURE_DESC_PATTERNS):
        return "public_figure"

    if username in POLITICAL_ACTOR_USERNAMES or contains_any(desc, POLITICAL_DESC_PATTERNS):
        return "media"

    if username in MEDIA_USERNAMES or contains_any(desc, MEDIA_DESC_PATTERNS):
        return "media"

    if contains_any(desc, OFFICIAL_DESC_PATTERNS):
        return "government_institution"

    if is_automated:
        return "automated_account"

    if followers >= 50000 and following <= 1000 and contains_any(desc, MEDIA_DESC_PATTERNS + OFFICIAL_DESC_PATTERNS + POLITICAL_DESC_PATTERNS + PUBLIC_FIGURE_DESC_PATTERNS):
        return "media_or_institution_like"

    return "citizen_candidate"

def classify_topic_relevance(row):
    text = norm_text(row.get("text"))
    keyword = norm_text(row.get("_search_keyword"))
    username = norm_text(row.get("author_username"))

    if username in IRRELEVANT_FESS_USERNAMES or contains_any(text, IRRELEVANT_TERMS):
        return "not_relevant"

    if contains_any(text, EDUCATION_TERMS) or contains_any(keyword, EDUCATION_TERMS):
        return "education_relevant"

    return "needs_manual_review"

def inclusion_decision(row):
    account_class = row["account_class"]
    topic = row["topic_relevance"]

    if account_class in {"media", "government_institution", "public_figure", "media_or_institution_like"}:
        return "exclude_context_source"
    if account_class == "automated_account":
        return "exclude_automated_account"
    if account_class == "irrelevant_fess" or topic == "not_relevant":
        return "exclude_not_relevant"
    if account_class == "education_fess_community" and topic == "education_relevant":
        return "include_education_community"
    if account_class == "general_fess" and topic == "education_relevant":
        return "include_with_review"
    if account_class == "citizen_candidate" and topic == "education_relevant":
        return "include_citizen_candidate"
    return "manual_review"

def source_weight(decision):
    if decision == "include_citizen_candidate":
        return 1.0
    if decision == "include_education_community":
        return 0.75
    if decision == "include_with_review":
        return 0.5
    return 0.0

## 5. Apply Rules

In [ ]:
df["account_class"] = df.apply(classify_account, axis=1)
df["topic_relevance"] = df.apply(classify_topic_relevance, axis=1)
df["verification_decision"] = df.apply(inclusion_decision, axis=1)
df["source_weight"] = df["verification_decision"].apply(source_weight)

summary = (
    df.groupby(["account_class", "topic_relevance", "verification_decision"])
    .size()
    .reset_index(name="n")
    .sort_values("n", ascending=False)
)

summary

## 6. Before vs After Comparison

In [ ]:
include_decisions = {
    "include_citizen_candidate",
    "include_education_community",
    "include_with_review",
}

df_before = df.copy()
df_after = df[df["verification_decision"].isin(include_decisions)].copy()

def numeric_col(data, column):
    if column not in data.columns:
        return pd.Series(dtype="float64")
    return pd.to_numeric(data[column], errors="coerce")

def dataset_profile(data, label):
    followers = numeric_col(data, "author_followers")
    likes = numeric_col(data, "likeCount")
    retweets = numeric_col(data, "retweetCount")
    replies = numeric_col(data, "replyCount")
    views = numeric_col(data, "viewCount")

    return {
        "dataset": label,
        "n_tweets": len(data),
        "n_unique_accounts": data["author_username"].nunique(dropna=True),
        "mean_followers": followers.mean(),
        "median_followers": followers.median(),
        "p75_followers": followers.quantile(0.75),
        "p90_followers": followers.quantile(0.90),
        "max_followers": followers.max(),
        "mean_likes": likes.mean(),
        "median_likes": likes.median(),
        "mean_retweets": retweets.mean(),
        "median_retweets": retweets.median(),
        "mean_replies": replies.mean(),
        "median_replies": replies.median(),
        "mean_views": views.mean(),
        "median_views": views.median(),
    }

comparison_profile = pd.DataFrame([
    dataset_profile(df_before, "before_raw"),
    dataset_profile(df_after, "after_analysis_ready"),
])

comparison_profile["retained_pct"] = [100, len(df_after) / len(df_before) * 100]
comparison_profile

In [ ]:
print("Jumlah data sebelum vs sesudah:")
print(f"Before/raw           : {len(df_before):,} tweet")
print(f"After/analysis-ready : {len(df_after):,} tweet")
print(f"Data retained        : {len(df_after) / len(df_before) * 100:.2f}%")
print(f"Data removed         : {(1 - len(df_after) / len(df_before)) * 100:.2f}%")

print("\nMean/median followers:")
comparison_profile[[
    "dataset", "mean_followers", "median_followers", "p75_followers",
    "p90_followers", "max_followers", "retained_pct"
]]

In [ ]:
def compare_distribution(column):
    before = df_before[column].value_counts(dropna=False).rename("before_n")
    after = df_after[column].value_counts(dropna=False).rename("after_n")
    out = pd.concat([before, after], axis=1).fillna(0).astype(int)
    out["before_pct"] = out["before_n"] / out["before_n"].sum() * 100
    out["after_pct"] = out["after_n"] / max(out["after_n"].sum(), 1) * 100
    out["retained_pct"] = out["after_n"] / out["before_n"].replace(0, pd.NA) * 100
    return out.sort_values("before_n", ascending=False)

region_comparison = compare_distribution("_region")
keyword_comparison = compare_distribution("_search_keyword")
account_class_comparison = compare_distribution("account_class")
decision_comparison = compare_distribution("verification_decision")

region_comparison

In [ ]:
keyword_comparison

In [ ]:
account_class_comparison

In [ ]:
decision_comparison

## 7. Check High-Frequency Accounts

In [ ]:
top_accounts = (
    df.groupby(["author_username", "account_class", "verification_decision"], dropna=False)
    .agg(
        n_tweets=("id", "count"),
        followers=("author_followers", "max"),
        following=("author_following", "max"),
        sample_keyword=("_search_keyword", "first"),
    )
    .reset_index()
    .sort_values("n_tweets", ascending=False)
)

top_accounts.head(30)

## 7.1 Audit High-Follower Citizen Candidates

In [ ]:
HIGH_FOLLOWER_THRESHOLD = 50_000

high_follower_citizen_candidates = (
    df[(df["account_class"] == "citizen_candidate") & (df["author_followers"] >= HIGH_FOLLOWER_THRESHOLD)]
    .groupby(["author_username", "author_name"], dropna=False)
    .agg(
        n_tweets=("id", "count"),
        followers=("author_followers", "max"),
        following=("author_following", "max"),
        sample_keyword=("_search_keyword", "first"),
        sample_bio=("author_description", "first"),
    )
    .reset_index()
    .sort_values(["n_tweets", "followers"], ascending=False)
)

print(f"High-follower citizen candidates needing manual audit: {len(high_follower_citizen_candidates):,} accounts")
high_follower_citizen_candidates.head(50)

## 8. Save Verified Dataset

In [ ]:
output_all = Path("verified_tweets_jakarta_banten_rule_based.csv")
output_included = Path("analysis_ready_tweets_jakarta_banten.csv")
output_summary = Path("verification_rule_summary.csv")
output_comparison_profile = Path("before_after_profile_comparison.csv")
output_region_comparison = Path("before_after_region_comparison.csv")
output_keyword_comparison = Path("before_after_keyword_comparison.csv")
output_account_class_comparison = Path("before_after_account_class_comparison.csv")
output_decision_comparison = Path("before_after_decision_comparison.csv")
output_high_follower_audit = Path("high_follower_citizen_audit.csv")

include_decisions = {
    "include_citizen_candidate",
    "include_education_community",
    "include_with_review",
}

analysis_ready = df[df["verification_decision"].isin(include_decisions)].copy()

# Output utama tetap disimpan untuk kompatibilitas dengan notebook lanjutan.
df.to_csv(output_all, index=False)
analysis_ready.to_csv(output_included, index=False)
summary.to_csv(output_summary, index=False)
comparison_profile.to_csv(output_comparison_profile, index=False)
region_comparison.to_csv(output_region_comparison)
keyword_comparison.to_csv(output_keyword_comparison)
account_class_comparison.to_csv(output_account_class_comparison)
decision_comparison.to_csv(output_decision_comparison)
high_follower_citizen_candidates.to_csv(output_high_follower_audit, index=False)

# Output rapi: pisahkan tweet inti, metadata akun, label, dan ringkasan.
organized_dir = Path("organized_csv")
organized_dir.mkdir(exist_ok=True)

tweet_core_columns = [
    "id", "createdAt", "_region", "_search_keyword", "text_clean", "url", "twitterUrl",
    "lang", "retweetCount", "replyCount", "likeCount", "quoteCount", "viewCount",
    "bookmarkCount", "isReply", "isQuote", "conversationId", "inReplyToUsername",
    "author_username",
]

author_columns = [
    "author_username", "author_name", "author_followers", "author_following",
    "author_statuses_count", "author_description_clean", "author_location",
    "author_is_blue_verified", "author_is_automated", "n_tweets",
]

label_columns = [
    "id", "author_username", "_region", "_search_keyword", "account_class",
    "topic_relevance", "verification_decision", "source_weight",
]

def clean_text_column(series):
    return (
        series.fillna("")
        .astype(str)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

def with_clean_columns(data):
    out = data.copy()
    out["text_clean"] = clean_text_column(out["text"] if "text" in out.columns else pd.Series(index=out.index, dtype="object"))
    out["author_description_clean"] = clean_text_column(out["author_description"] if "author_description" in out.columns else pd.Series(index=out.index, dtype="object"))
    return out

def ensure_columns(data, columns):
    out = data.copy()
    for column in columns:
        if column not in out.columns:
            out[column] = pd.NA
    return out[columns]

def make_authors_table(data):
    cleaned = with_clean_columns(data)
    base_cols = [col for col in author_columns if col != "n_tweets"]
    authors = ensure_columns(cleaned, base_cols).drop_duplicates(subset=["author_username"], keep="first")
    counts = cleaned["author_username"].value_counts(dropna=False).rename("n_tweets")
    authors = authors.merge(counts, left_on="author_username", right_index=True, how="left")
    return authors[author_columns].sort_values("n_tweets", ascending=False)

def save_organized_dataset(data, folder, include_labels):
    folder.mkdir(parents=True, exist_ok=True)
    cleaned = with_clean_columns(data)

    ensure_columns(cleaned, tweet_core_columns).to_csv(folder / "tweets_core.csv", index=False)
    make_authors_table(cleaned).to_csv(folder / "authors.csv", index=False)

    if include_labels:
        ensure_columns(cleaned, label_columns).to_csv(folder / "verification_labels.csv", index=False)

        by_decision = folder / "by_verification_decision"
        by_decision.mkdir(exist_ok=True)
        for decision, group in cleaned.groupby("verification_decision", dropna=False):
            safe_name = str(decision).replace("/", "_").replace("\\", "_")
            cols = tweet_core_columns + ["account_class", "topic_relevance", "verification_decision", "source_weight"]
            ensure_columns(group, cols).to_csv(by_decision / f"{safe_name}.csv", index=False)

save_organized_dataset(df, organized_dir / "02_verified", include_labels=True)
save_organized_dataset(analysis_ready, organized_dir / "03_analysis_ready", include_labels=True)

# Folder raw juga dibuat dari df penuh, tetapi hanya kolom inti dan akun agar tidak berantakan.
save_organized_dataset(df, organized_dir / "01_raw", include_labels=False)

comparison_dir = organized_dir / "04_comparison"
comparison_dir.mkdir(exist_ok=True)
summary.to_csv(comparison_dir / "verification_rule_summary.csv", index=False)
comparison_profile.to_csv(comparison_dir / "before_after_profile_comparison.csv", index=False)
region_comparison.to_csv(comparison_dir / "before_after_region_comparison.csv")
keyword_comparison.to_csv(comparison_dir / "before_after_keyword_comparison.csv")
account_class_comparison.to_csv(comparison_dir / "before_after_account_class_comparison.csv")
decision_comparison.to_csv(comparison_dir / "before_after_decision_comparison.csv")
high_follower_citizen_candidates.to_csv(comparison_dir / "high_follower_citizen_audit.csv", index=False)

(organized_dir / "README.md").write_text(
    "# Organized CSV Outputs\n\n"
    "01_raw berisi tweet inti dan akun sebelum pemilahan akhir.\n"
    "02_verified berisi dataset lengkap dengan label rule-based.\n"
    "03_analysis_ready berisi data final yang siap untuk analisis sentimen/topik.\n"
    "04_comparison berisi ringkasan before/after dan distribusi filter.\n",
    encoding="utf-8",
)

print(f"Saved full labeled dataset: {output_all}")
print(f"Saved analysis-ready dataset: {output_included}")
print(f"Saved summary: {output_summary}")
print(f"Saved organized CSV folder: {organized_dir}")
print(f"Included rows: {len(analysis_ready):,} / {len(df):,}")

## Methodology Note

Rule-based verification digunakan sebagai lapisan awal untuk memisahkan sumber informasi dari aspirasi warga. Akun media, aktor politik, public figure, akun otomatis, dan institusi pemerintah tidak dihapus dari dataset berlabel, tetapi dikeluarkan dari analisis aspirasi publik karena lebih berfungsi sebagai sumber konteks, aktor institusional, komunikasi elite, atau kanal otomatis. Akun fess bertema pendidikan seperti komunitas SNBT, UTBK, sekolah, dan kampus tetap dipertahankan dengan bobot lebih rendah karena merepresentasikan ruang percakapan pelajar/mahasiswa, bukan individu tunggal. Fess umum dan fess tidak relevan dipisahkan melalui filter relevansi topik agar tidak mencampurkan noise non-pendidikan ke analisis sentimen. Selain itu, akun yang masih lolos sebagai citizen_candidate tetapi memiliki followers tinggi diaudit manual sebagai quality-control untuk mendeteksi media, public figure, atau base yang belum masuk whitelist.